# Tracker Sensitivity Analysis — ByteTrack Parameter Study (Only for Analysis, Not Primary Experiment)

**Scope:** Side experiment only. Primary experiment (notebooks 01–03) uses Ultralytics
ByteTrack defaults unchanged. This notebook quantifies how IDSW, fragmentation, and MT ratio
change when one parameter is varied at a time, informing future deployment guidance.

**Fixed:** model = yolo26s (selected in Exp1), resolution = 640px, all three MOT17 sequences.

**Varied (one at a time, others at default):**
- `track_buffer`: 30 (default) → 60  — addresses MOT17-04 fragmentation (lost-track recovery)
- `track_high_thresh`: 0.25 (default) → 0.4, 0.5  — addresses MOT17-09 IDSW (prune noisy dets)
- `match_thresh`: 0.8 (default) → 0.7, 0.9  — cross-sequence association tightness

**Not changed:** `track_low_thresh`, `new_track_thresh`, `fuse_score` (insufficient motivation).

**Methodology note:** These runs are independent from the primary experiment data. Results
are compared against the primary experiment baseline (yolo26s, 640px, default params).


In [ ]:
# Configuration
import tempfile, yaml, sys
from pathlib import Path
import torch

sys.path.insert(0, str(Path('.').resolve().parent / 'src'))
from benchmark.config import DATA_ROOT, RESULTS_RAW, SEQ_SUFFIX, SEQUENCES
from benchmark.runner import run_sequence
from benchmark.metrics import compute_mot_metrics

SELECTED_MODEL = "yolo26s.pt"
DEVICE         = "cuda:0" if torch.cuda.is_available() else "cpu"
SENS_DIR       = RESULTS_RAW.parent / "sensitivity"
SENS_DIR.mkdir(parents=True, exist_ok=True)
SKIP_EXISTING  = True

# ByteTrack defaults (primary experiment baseline)
BYTETRACK_DEFAULTS = {
    "tracker_type":     "bytetrack",
    "track_high_thresh": 0.25,
    "track_low_thresh":  0.1,
    "new_track_thresh":  0.25,
    "track_buffer":      30,
    "match_thresh":      0.8,
    "fuse_score":        True,
}

# One-at-a-time parameter grid
PARAM_GRID = [
    {"name": "default",          "params": {}},
    {"name": "buffer_60",        "params": {"track_buffer": 60}},
    {"name": "high_thresh_0.4",  "params": {"track_high_thresh": 0.4}},
    {"name": "high_thresh_0.5",  "params": {"track_high_thresh": 0.5}},
    {"name": "match_thresh_0.7", "params": {"match_thresh": 0.7}},
    {"name": "match_thresh_0.9", "params": {"match_thresh": 0.9}},
]

print(f"Sensitivity configs: {len(PARAM_GRID)}")
print(f"Sequences: {SEQUENCES}")
print(f"Total runs: {len(PARAM_GRID)} × {len(SEQUENCES)} = {len(PARAM_GRID)*len(SEQUENCES)}")


In [ ]:
# Inference loop: one config × one sequence per cell execution
# A custom bytetrack YAML is written per config variant; model is re-instantiated
# per (config, sequence) to ensure clean ByteTrack state at every run boundary.
from ultralytics import YOLO

def _make_tracker_yaml(overrides: dict) -> str:
    """Write a bytetrack YAML with parameter overrides; return its path."""
    cfg = {**BYTETRACK_DEFAULTS, **overrides}
    with tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", delete=False) as f:
        yaml.dump(cfg, f)
        return f.name

for cfg in PARAM_GRID:
    cfg_name     = cfg["name"]
    overrides    = cfg["params"]
    tracker_yaml = _make_tracker_yaml(overrides)

    print(f"\n── Config: {cfg_name}  overrides={overrides} ──")
    for seq_name in SEQUENCES:
        out_csv = SENS_DIR / f"{seq_name}_{SELECTED_MODEL}_640_{cfg_name}.csv"
        if SKIP_EXISTING and out_csv.exists():
            print(f"  {seq_name}: skip")
            continue

        seq_dir = DATA_ROOT / f"{seq_name}-{SEQ_SUFFIX}"
        model   = YOLO(SELECTED_MODEL)
        model.to(DEVICE)

        print(f"  {seq_name}: running ...", end=" ", flush=True)
        # Pass the custom tracker YAML directly; run_sequence uses it via the tracker= param
        df = run_sequence(model, seq_dir, imgsz=640, out_csv=out_csv, tracker=tracker_yaml)
        print(f"done — mean {df['n_detections'].mean():.1f} det/frame")

In [ ]:
# Compute MOT metrics for every (config × sequence) run
import pandas as pd

rows = []
for cfg in PARAM_GRID:
    cfg_name = cfg["name"]
    overrides = cfg["params"]

    for seq_name in SEQUENCES:
        # Sensitivity runs
        p = SENS_DIR / f"{seq_name}_{SELECTED_MODEL}_640_{cfg_name}.csv"
        if not p.exists():
            continue
        seq_dir = DATA_ROOT / f"{seq_name}-{SEQ_SUFFIX}"
        m = compute_mot_metrics(p, seq_dir)
        rows.append({
            "config":       cfg_name,
            "override":     str(overrides) if overrides else "—",
            "seq":          seq_name,
            "mota":         round(m["mota"], 3),
            "idf1":         round(m["idf1"], 3),
            "idsw":         m["num_switches"],
            "idsw/gt":      round(m["idsw_per_gt_track"], 3),
            "frag":         round(m["frag_ratio"], 3),
            "tot_init":     m["total_initiated"],
            "mostly_tracked": m["mostly_tracked"],
        })
        print(f"  {cfg_name} / {seq_name}: MOTA={m['mota']:.3f}  IDSW={m['num_switches']}  IDSW/GT={m['idsw_per_gt_track']:.3f}  frag={m['frag_ratio']:.3f}")

sens_df = pd.DataFrame(rows)
sens_df.to_csv(SENS_DIR / "sensitivity_metrics.csv", index=False)
print("\nSaved sensitivity_metrics.csv")


In [ ]:
# Delta table: change vs default config per (sequence, metric)
# Positive delta = improvement; sign is set so higher = better for all metrics.
import numpy as np

pd.set_option("display.width", 160)

delta_rows = []
for seq_name in SEQUENCES:
    base = sens_df[(sens_df["config"] == "default") & (sens_df["seq"] == seq_name)]
    if base.empty:
        continue
    b_mota, b_idf1, b_idsw, b_frag = (
        float(base["mota"].iloc[0]),
        float(base["idf1"].iloc[0]),
        float(base["idsw/gt"].iloc[0]),
        float(base["frag"].iloc[0]),
    )

    for _, row in sens_df[sens_df["seq"] == seq_name].iterrows():
        delta_rows.append({
            "config":        row["config"],
            "seq":           seq_name,
            "ΔMOTA":         round(row["mota"] - b_mota, 3),
            "ΔIDF1":         round(row["idf1"] - b_idf1, 3),
            "ΔIDSW/GT":      round(b_idsw - row["idsw/gt"], 3),   # positive = fewer switches
            "ΔFrag":         round(b_frag - row["frag"], 3),       # positive = less fragmentation
            "MT":            row["mostly_tracked"],
        })

delta_df = pd.DataFrame(delta_rows)
print("Delta vs default (positive = improvement):")
print(delta_df.to_string(index=False))


In [ ]:
# Heatmap: per-config improvement per sequence across three signals
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

signals = ["ΔIDSW/GT", "ΔFrag", "ΔMOTA"]
configs = [c["name"] for c in PARAM_GRID if c["name"] != "default"]

fig, axes = plt.subplots(1, len(SEQUENCES), figsize=(12, 3.5), sharey=True)

for ax, seq_name in zip(axes, SEQUENCES):
    sub  = delta_df[delta_df["seq"] == seq_name]
    data = []
    for cfg_name in configs:
        row = sub[sub["config"] == cfg_name]
        if row.empty:
            data.append([0, 0, 0])
        else:
            data.append([float(row[s].iloc[0]) for s in signals])

    mat = np.array(data)  # (n_configs, n_signals)

    # Diverging colour: green = improvement, red = regression
    vmax = max(abs(mat).max(), 0.05)
    im = ax.imshow(mat, cmap="RdYlGn", vmin=-vmax, vmax=vmax, aspect="auto")

    ax.set_xticks(range(len(signals)))
    ax.set_xticklabels(signals, fontsize=9)
    ax.set_yticks(range(len(configs)))
    if ax is axes[0]:
        ax.set_yticklabels(configs, fontsize=8)
    ax.set_title(seq_name, fontsize=10, fontweight="bold")

    # Annotate cells with the delta value
    for i, cfg_name in enumerate(configs):
        for j, sig in enumerate(signals):
            ax.text(j, i, f"{mat[i, j]:+.2f}", ha="center", va="center", fontsize=8,
                    color="black" if abs(mat[i, j]) < vmax * 0.6 else "white")

plt.colorbar(im, ax=axes[-1], fraction=0.04, pad=0.04, label="Δ vs default (positive = better)")
fig.suptitle(
    f"ByteTrack parameter sensitivity — {SELECTED_MODEL}, 640px — Δ vs default config",
    fontsize=10, fontweight="bold", y=1.02,
)
plt.tight_layout()

fig.savefig(SENS_DIR / "fig_sensitivity.pdf", bbox_inches="tight", dpi=300)
fig.savefig(SENS_DIR / "fig_sensitivity.png", bbox_inches="tight", dpi=300)
print("Saved: fig_sensitivity.pdf / .png")
plt.show()


## Interpretation guide

**`track_buffer` 30 → 60** (`buffer_60`):  
Doubles the window in which a lost track can be re-associated before it is terminated.
Expected benefit: fewer fragmented tracks on MOT17-04 (dense, long GT tracks with median 468 frames).
Expected risk: may increase IDSW if re-associations match wrong returning detections.

**`track_high_thresh` 0.25 → 0.4 / 0.5** (`high_thresh_0.4`, `high_thresh_0.5`):  
Raises the confidence floor for first-stage association. On MOT17-09, >80% of detections
have conf ≥ 0.46 — raising the threshold prunes the ambiguous low-confidence pool that
creates spurious IoU matches at the low-angle viewing geometry.  
Expected benefit: reduced IDSW on MOT17-09.  
Expected risk: may lose valid tracks for partially occluded pedestrians (lower recall).

**`match_thresh` 0.8 → 0.7 / 0.9** (`match_thresh_0.7`, `match_thresh_0.9`):  
Controls the IoU cost gate for the assignment solver. A lower gate (0.7) rejects more
candidate matches → fewer erroneous associations but possibly lower recall. A higher gate
(0.9) accepts more matches → better recall but increased ambiguity in crowds.

**Reading the heatmap:**  
Green = improvement relative to default. A parameter that is beneficial for one sequence
but regresses another is a confounding factor and should **not** be adopted as a universal default.
